**PADDLEOCR fine tune recognition model (2025/12/3) using synthetic dataset** epoch 25 and lr 0.0005

**Step 1: Mount Google Drive and Install PaddlecOCR**

In [1]:
!nvidia-smi

Wed Dec 31 06:29:56 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [4]:
!python -m pip install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu118/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 GB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 14.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 699.9/699.9 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 12.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
# Install Compatible PyTorch
!pip uninstall paddleocr paddlepaddle torch torchvision torchaudio -y
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu118

Found existing installation: torch 2.9.0+cu126
Uninstalling torch-2.9.0+cu126:
  Successfully uninstalled torch-2.9.0+cu126
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126
Found existing installation: torchaudio 2.9.0+cu126
Uninstalling torchaudio-2.9.0+cu126:
  Successfully uninstalled torchaudio-2.9.0+cu126
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.6/839.6 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 147.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 129.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.9/142.9 MB 15.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nccl-cu11
    Found existing installation: nvidia-nccl-cu11 2.19.3
    Uninstalling nvidia-nccl-cu1

In [13]:
!pip install -r requirements.txt
!pip install paddleocr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 141.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.1.20 requires numpy<2,>=1, but you have numpy 2.2.6 which is incompatible.
langchain-community 0.0.38 requires numpy<2,>=1, but you have numpy 2.2.6 which is incompatible.
paddlepaddle-gpu 3.2.0 requires nvidia-cudnn-cu11==8.9.6.50; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn-cu11 8.7.0.84 which is i

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
^C


In [1]:
# Install compatible langchain and restart runtime
!pip install "langchain<0.2.0"

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
paddlepaddle-gpu 3.2.0 requires nvidia-cudnn-cu11==8.9.6.50; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn-cu11 8.7.0.84 which is incompatible.
paddlepaddle-gpu 3.2.0 requires nvidia-nccl-cu11==2.19.3; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-nccl-cu11 2.20.5 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, 

In [1]:
# Ensure paddleocr is successfully installed
import paddle
paddle.utils.run_check()
print("CUDA:", paddle.version.cuda())  # 12.6
print("GPU compiled:", paddle.is_compiled_with_cuda())  # True
print("Device:", paddle.device.get_device())  # gpu:0
import torch
print("Torch CUDA:", torch.cuda.is_available())  # True
print("NCCL version:", torch.cuda.nccl.version())  # ~2.27.5

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Running verify PaddlePaddle program ... 


/usr/local/lib/python3.12/dist-packages/paddle/pir/math_op_patch.py:219: UserWarning: Value do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(


PaddlePaddle works well on 1 GPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.
CUDA: 11.8
GPU compiled: True
Device: gpu:0
Torch CUDA: True
NCCL version: (2, 20, 5)


In [1]:
%cd /content
!rm -rf PaddleOCR

/content


In [3]:
!git clone -b release/3.0 https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR

Cloning into 'PaddleOCR'...
remote: Enumerating objects: 308397, done.
remote: Counting objects: 100% (871/871), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 308397 (delta 754), reused 747 (delta 693), pack-reused 307526 (from 2)
Receiving objects: 100% (308397/308397), 1.63 GiB | 39.43 MiB/s, done.
Resolving deltas: 100% (244123/244123), done.
/content/PaddleOCR


**Step 2: Import the Dataset**

In [5]:
!ln -s /content/drive/MyDrive/MyInvoiceDataset /content/PaddleOCR/invoice_dataset

In [6]:
!head /content/PaddleOCR/invoice_dataset/train_labels.txt

0.jpg	TacoSeasoning
1.jpg	LaundryDetergent包廂逾期費用
2.jpg	牙膏
3.jpg	用量Apples
4.jpg	CoinsCrackers
5.jpg	Baker’sBroadway
6.jpg	空調使用Bread
7.jpg	啤酒phonez
8.jpg	空調使用changethankyou
9.jpg	地下停車場從Kleenex


**Step 3: Download the dictionary file and pretrianed model**

In [6]:
!wget -P /content/ https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/release/3.0/ppocr/utils/dict/chinese_cht_dict.txt

--2025-12-31 06:41:31--  https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/release/3.0/ppocr/utils/dict/chinese_cht_dict.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 33443 (33K) [text/plain]
Saving to: ‘/content/chinese_cht_dict.txt’

chinese_cht_dict.tx 100%[===================>]  32.66K  --.-KB/s    in 0.003s  

2025-12-31 06:41:31 (10.4 MB/s) - ‘/content/chinese_cht_dict.txt’ saved [33443/33443]



In [7]:
!mkdir -p /content/pretrain_models
!wget -P /content/pretrain_models/ https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_rec_pretrained.pdparams

--2025-12-31 06:41:36--  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_rec_pretrained.pdparams
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:628:0:ff:b0e8:88da
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 214594738 (205M) [application/octet-stream]
Saving to: ‘/content/pretrain_models/PP-OCRv5_server_rec_pretrained.pdparams’

PP-OCRv5_server_rec 100%[===================>] 204.65M  13.9MB/s    in 32s     

2025-12-31 06:42:09 (6.39 MB/s) - ‘/content/pretrain_models/PP-OCRv5_server_rec_pretrained.pdparams’ saved [214594738/214594738]



**Optional: Download the config file for editing**

In [8]:
import shutil
shutil.copy('/content/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml', '/content/ch_invoice_rec.yml')

'/content/ch_invoice_rec.yml'

**Step 4: Edit the config file ch_invoice_rec.yml then upload in Colab**

In [9]:
%cd /content

/content


In [10]:
from google.colab import files

print("Please select a file to upload:")
uploaded = files.upload()

if not uploaded:
    print("Alert: No file was selected for upload. Please ensure you choose a file.")
else:
    for filename in uploaded.keys():
        print(f"Alert: File '{filename}' has been successfully uploaded.")

Please select a file to upload:


Saving ch_invoice_rec.yml to ch_invoice_rec.yml
Alert: File 'ch_invoice_rec.yml' has been successfully uploaded.


**Step 5: Start Training and Export the inference model**

In [2]:
%cd /content/PaddleOCR

[Errno 2] No such file or directory: '/content/PaddleOCR'
/content


In [7]:
!python tools/train.py -c /content/ch_invoice_rec.yml -o Global.use_gpu=True

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2025/12/31 06:50:39] ppocr WARNING: Skipping import of the encryption module.
[2025/12/31 06:50:39] ppocr INFO: Architecture : 
[2025/12/31 06:50:39] ppocr INFO:     Backbone : 
[2025/12/31 06:50:39] ppocr INFO:         name : PPHGNetV2_B4
[2025/12/31 06:50:39] ppocr INFO:         text_rec : True
[2025/12/31 06:50:39] ppocr INFO:     Head : 
[2025/12/31 06:50:39] ppocr INFO:         head_list : 
[2025/12/31 06:50:39] ppocr INFO:             CTCHead : 
[2025/12/31 06:50:39] ppocr INFO:                 Head : 
[2025/12/31 06:50:39] ppocr INFO:                     fc_decay : 1e-05
[2025/12/31 06:50:39] ppocr INFO:                 Neck : 
[2025/12/31 06:50:39] ppocr INFO:

In [8]:
!python tools/export_model.py -c /content/ch_invoice_rec.yml -o Global.pretrained_model=/content/output/invoice_rec/best_accuracy Global.save_inference_dir=/content/invoice_rec_inference

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2025/12/31 07:14:32] ppocr WARNING: Skipping import of the encryption module.
W1231 07:14:33.905936 12919 gpu_resources.cc:114] Please NOTE: device: 0, GPU Compute Capability: 8.0, Driver API Version: 12.4, Runtime API Version: 11.8
[2025/12/31 07:14:34] ppocr INFO: load pretrain successful from /content/output/invoice_rec/best_accuracy
[2025/12/31 07:14:34] ppocr INFO: Export inference config file to /content/invoice_rec_inference/inference.yml
Skipping import of the encryption module
W1231 07:14:36.493885 12919 eager_utils.cc:3441] Paddle static graph(PIR) not support input out tensor for now!!!!!
[2025/12/31 07:14:37] ppocr INFO: inference model is saved to /conten

In [9]:
#Do not copy invoice_rec if not enough disk space
#!cp -r /content/output/invoice_rec /content/drive/MyDrive/
!cp -r /content/invoice_rec_inference /content/drive/MyDrive/

**Step 6: Run Inference using fine-tuned recognition model stored in the folder invoice_rec_inference**

In [10]:
!mkdir -p ./inference_results ./output/det_rec

In [12]:
!paddleocr ocr -i /content/drive/MyDrive/test_images --text_recognition_model_dir /content/invoice_rec_inference --device gpu --save_path ./inference_results --lang ch --ocr_version PP-OCRv5

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
/usr/local/lib/python3.12/dist-packages/paddleocr/_utils/cli.py:62: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  wrapper = wrapper_cls(**init_params)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory man

**Step 7: Copy Inference Result**

In [13]:
!cp -r /content/PaddleOCR/inference_results /content/drive/MyDrive/